# MaiForge / mainote Linux 快速启动（Web UI）

这个 notebook 面向 **Linux**，按顺序完成：

1. 检查仓库位置、Python 和 `ffmpeg`
2. 创建 `.venv` 并安装 `requirements.txt`
3. 在后台启动 `mainote web`
4. 检查 `/api/health` 并打开 Web UI
5. 停止服务（可选）
6. 安装 PyTorch 2.8.0 + CUDA 12.8（可选，训练/推理）

> 请从第一个单元格开始依次运行。仓库已经提交了构建好的前端静态资源，**不需要安装 Node.js**。
>
> `requirements.txt` 只安装 Web/音频依赖，**不包含 PyTorch**；P3 训练和推理所需的
> PyTorch 2.8.0（CUDA 12.8）在最后一节单独安装。


## 0. 系统前置条件

推荐 **Python 3.12（Ubuntu 22.04）** 和 `ffmpeg`；项目最低要求 Python 3.11。
GPU 训练还需要支持 **CUDA 12.8** 的 NVIDIA 驱动。Debian / Ubuntu 可以先用 apt
安装系统包：

```bash
sudo apt-get update
sudo apt-get install -y python3-venv ffmpeg
```

如果 Jupyter / JupyterLab 运行在**远程服务器**上，Web UI 默认只监听 `127.0.0.1`，本机浏览器无法直接访问。请在自己电脑上建立 SSH 端口转发：

```bash
ssh -N -L 8765:127.0.0.1:8765 <user>@<server>
```

然后在本地浏览器打开 `http://127.0.0.1:8765`。也可以使用 VS Code / Jupyter 自带的端口转发。


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "mainote").is_dir():
            return candidate
    raise RuntimeError(
        "没有找到 MaiForge 仓库。请把 notebook 放在仓库内，"
        "或从仓库根目录 / notebooks 目录启动 Jupyter。"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

print(f"Python: {sys.version.split()[0]}")
print(f"仓库根目录: {REPO_ROOT}")

if not (REPO_ROOT / "src" / "mainote" / "webapp" / "static" / "index.html").is_file():
    print("警告: 未找到已构建的前端 static/index.html，Web UI 可能显示空白页。")


In [ ]:
import shutil
import subprocess

# Debian / Ubuntu 且当前用户能免密 sudo 时，可改成 True 自动安装。
# 如果 sudo 需要输入密码，请在终端里手动安装，避免 notebook 卡住。
INSTALL_FFMPEG_WITH_APT = False

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    print("未找到 ffmpeg。")
    if INSTALL_FFMPEG_WITH_APT:
        subprocess.run(["sudo", "apt-get", "update"], check=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "ffmpeg"], check=True)
        ffmpeg = shutil.which("ffmpeg")
else:
    print(f"ffmpeg: {ffmpeg}")

if not ffmpeg:
    raise RuntimeError(
        "请先安装 ffmpeg。Debian/Ubuntu: "
        "sudo apt-get update && sudo apt-get install -y ffmpeg"
    )

version = subprocess.run(
    [ffmpeg, "-version"],
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()[0]
print(version)


## 1. 创建虚拟环境并安装依赖

下面会在仓库根目录创建 `.venv`（如果已存在则复用），然后用它安装：

```text
requirements.txt  ->  -e .[audio,web]
```

Web UI、音频分析和在线谱库所需的依赖都会装进 `.venv`，不会污染当前 Jupyter kernel。

> 国内网络如果 pip 很慢，可以把代码里的 `PIP_INDEX_URL` 改成
> `https://pypi.tuna.tsinghua.edu.cn/simple`。


In [ ]:
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"需要 Python 3.11+，当前 Jupyter kernel 是 {sys.version.split()[0]}。"
    )
if sys.version_info[:2] != (3, 12):
    print(
        f"提示: PyTorch 2.8.0 + CUDA 12.8 组合按 Python 3.12 / Ubuntu 22.04 "
        f"验证；当前 Jupyter kernel 是 {sys.version.split()[0]}。"
    )

VENV_DIR = REPO_ROOT / ".venv"
VENV_PY = VENV_DIR / "bin" / "python"

# 国内网络可改成 "https://pypi.tuna.tsinghua.edu.cn/simple"
PIP_INDEX_URL = ""
PIP_INDEX_ARGS = ["-i", PIP_INDEX_URL] if PIP_INDEX_URL else []

if not VENV_PY.is_file():
    print(f"创建虚拟环境: {VENV_DIR}")
    try:
        subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "创建 venv 失败。Debian/Ubuntu 通常需要先安装 python3-venv："
            "sudo apt-get install -y python3-venv"
        ) from exc
else:
    print(f"复用已有虚拟环境: {VENV_DIR}")

print(subprocess.run(
    [str(VENV_PY), "--version"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip())

# 更新 pip 并安装 Web UI / 音频依赖（editable 安装）
subprocess.run(
    [str(VENV_PY), "-m", "pip", "install", "-U", "pip", *PIP_INDEX_ARGS],
    cwd=str(REPO_ROOT),
    check=True,
)
subprocess.run(
    [
        str(VENV_PY),
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO_ROOT / "requirements.txt"),
        *PIP_INDEX_ARGS,
    ],
    cwd=str(REPO_ROOT),
    check=True,
)

# 快速确认关键模块已安装
subprocess.run(
    [
        str(VENV_PY),
        "-c",
        "import fastapi, uvicorn, numpy, mutagen, yaml; "
        "print('依赖检查通过')",
    ],
    cwd=str(REPO_ROOT),
    check=True,
)


## 2. 在后台启动 Web UI

Web 服务会作为独立进程运行，notebook 单元格不会阻塞。服务启动后：

- 日志：`out/webui-notebook.log`
- PID：`out/webui-notebook.pid`
- 默认端口 `8765`；如果被占用，会自动尝试 `8766`–`8784`

默认只监听 `127.0.0.1`。如果确实需要让同一局域网访问，可以把下面代码里的
`BIND_HOST` 改成 `"0.0.0.0"`，并把 `ACCESS_HOST` 改成服务器 IP。
**Web UI 没有登录鉴权，只建议在可信网络中使用。**


In [ ]:
import os
import signal
import socket
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

# 监听地址；局域网访问可改为 "0.0.0.0"（无鉴权，慎用）
BIND_HOST = "127.0.0.1"
# 浏览器访问地址；SSH 隧道场景保持 127.0.0.1，局域网可填服务器 IP
ACCESS_HOST = "127.0.0.1"
PORT = 8765

OUT_DIR = REPO_ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "webui-notebook.log"
PID_PATH = OUT_DIR / "webui-notebook.pid"
VENV_PY = REPO_ROOT / ".venv" / "bin" / "python"

if not VENV_PY.is_file():
    raise RuntimeError("没有找到 .venv/bin/python，请先运行上面的“安装依赖”单元格。")


def _port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            sock.bind((BIND_HOST, port))
        except OSError:
            return False
    return True


def _choose_port(preferred: int) -> int:
    for port in range(preferred, preferred + 20):
        if _port_is_free(port):
            return port
    raise RuntimeError(f"{preferred}–{preferred + 19} 都被占用，请手动修改 PORT。")


def _pid_is_webui(pid: int) -> bool:
    try:
        args = subprocess.check_output(
            ["ps", "-p", str(pid), "-o", "args="],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return False
    return "mainote" in args and " web" in args


def _wait_for_exit(pid: int, timeout: float = 5.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            os.kill(pid, 0)
        except OSError:
            return True
        time.sleep(0.1)
    return False


def stop_webui() -> None:
    if not PID_PATH.is_file():
        print("没有记录在案的 Web UI 进程。")
        return

    try:
        pid = int(PID_PATH.read_text(encoding="utf-8").strip())
    except (OSError, ValueError):
        PID_PATH.unlink(missing_ok=True)
        print("PID 文件无效，已删除。")
        return

    if not _pid_is_webui(pid):
        PID_PATH.unlink(missing_ok=True)
        print(f"PID {pid} 不是本 notebook 启动的 mainote web，已清理 PID 文件。")
        return

    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

    if not _wait_for_exit(pid, timeout=5.0):
        try:
            os.killpg(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

    PID_PATH.unlink(missing_ok=True)
    print(f"已停止 Web UI（PID {pid}）。")


def start_webui(preferred_port: int = PORT) -> str:
    if PID_PATH.is_file():
        stop_webui()

    port = _choose_port(int(preferred_port))
    env = os.environ.copy()
    src = str(REPO_ROOT / "src")
    env["PYTHONPATH"] = src + (
        os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
    )

    log = open(LOG_PATH, "w", encoding="utf-8")
    try:
        proc = subprocess.Popen(
            [
                str(VENV_PY),
                "-m",
                "mainote",
                "web",
                "--host",
                BIND_HOST,
                "--port",
                str(port),
                "--no-browser",
            ],
            cwd=str(REPO_ROOT),
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    finally:
        log.close()

    PID_PATH.write_text(str(proc.pid), encoding="utf-8")
    url = f"http://{ACCESS_HOST}:{port}"
    health_url = f"http://127.0.0.1:{port}/api/health"
    deadline = time.time() + 60.0

    while time.time() < deadline:
        if proc.poll() is not None:
            text = LOG_PATH.read_text(encoding="utf-8", errors="replace")
            PID_PATH.unlink(missing_ok=True)
            raise RuntimeError(
                f"mainote web 启动失败（退出码 {proc.returncode}）。日志：\n{text[-5000:]}"
            )
        try:
            with urllib.request.urlopen(health_url, timeout=1.5) as response:
                if response.status == 200:
                    print(f"Web UI 已启动: {url}")
                    print(f"日志文件: {LOG_PATH}")
                    print(f"进程 PID: {proc.pid}")
                    return url
        except OSError:
            pass
        time.sleep(0.5)

    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    PID_PATH.unlink(missing_ok=True)
    text = LOG_PATH.read_text(encoding="utf-8", errors="replace")
    raise RuntimeError(f"启动超时（60 秒）。最近日志：\n{text[-5000:]}")


In [ ]:
from IPython.display import Markdown, display

WEBUI_URL = start_webui(PORT)
display(Markdown(f"### Web UI 已就绪\n\n[{WEBUI_URL}]({WEBUI_URL})\n\n如果页面空白，请稍等几秒后刷新。"))


In [ ]:
import json
from urllib.parse import urlsplit

port = urlsplit(WEBUI_URL).port
with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/health", timeout=5) as response:
    health = json.load(response)

print("健康检查:", json.dumps(health, ensure_ascii=False))
print("\n最近日志:")
for line in LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-25:]:
    print(line)


## 3. 可选：生成演示数据

如果还没有导入任何歌曲，可以生成一份合成演示数据，随后在 Web UI 的“全部曲库”中查看。
把下面的 `RUN_DEMO` 改成 `True` 再运行即可。


In [ ]:
RUN_DEMO = False

if RUN_DEMO:
    subprocess.run(
        [
            str(VENV_PY),
            str(REPO_ROOT / "scripts" / "make_demo.py"),
            str(REPO_ROOT / "out" / "demo"),
        ],
        cwd=str(REPO_ROOT),
        check=True,
    )
    print("演示数据已生成。刷新 Web UI 的“全部曲库”即可看到。")
else:
    print("跳过演示数据。将 RUN_DEMO 改为 True 可生成合成数据。")


## 4. 停止 Web UI

默认 `STOP_WEBUI = False`，这样即使点“Run All”也不会把刚启动的服务立刻关掉。
需要停止时，把下面的 `STOP_WEBUI` 改成 `True`，再单独运行这个单元格。


In [ ]:
STOP_WEBUI = False

if STOP_WEBUI:
    stop_webui()
else:
    print("Web UI 保持运行。需要停止时，把 STOP_WEBUI 改为 True 后重新运行本单元格。")


## 5. 可选：安装 PyTorch（CUDA 12.8）

P3 训练 / 推理需要 PyTorch。下面的组合固定为 **PyTorch 2.8.0 + CUDA 12.8**，
适用于 **Python 3.12 / Ubuntu 22.04** 的 NVIDIA GPU 服务器：

```bash
.venv/bin/python -m pip install \
  torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 \
  --index-url https://download.pytorch.org/whl/cu128
.venv/bin/python -m pip install -r requirements-train.txt
```

把下一格的 `INSTALL_PYTORCH` 改成 `True` 即可执行。该单元格会先安装 CUDA 12.8
版本的 PyTorch，再安装 `requirements-train.txt`，最后打印 `torch.__version__`
和 `torch.version.cuda` 做确认。CPU-only 环境请把 `PYTORCH_INDEX_URL` 改为
`https://download.pytorch.org/whl/cpu`，并去掉 `torchvision` / `torchaudio` 的
CUDA 版本假设。


In [ ]:
# PyTorch 2.8.0 + CUDA 12.8（Python 3.12 / Ubuntu 22.04）
INSTALL_PYTORCH = False

PYTORCH_INDEX_URL = "https://download.pytorch.org/whl/cu128"
PYTORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]

if INSTALL_PYTORCH:
    subprocess.run(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            *PYTORCH_PACKAGES,
            "--index-url",
            PYTORCH_INDEX_URL,
        ],
        cwd=str(REPO_ROOT),
        check=True,
    )
    subprocess.run(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "-r",
            str(REPO_ROOT / "requirements-train.txt"),
        ],
        cwd=str(REPO_ROOT),
        check=True,
    )
    subprocess.run(
        [
            str(VENV_PY),
            "-c",
            "import torch; "
            "print('torch', torch.__version__, "
            "'| CUDA', torch.version.cuda, "
            "'| available', torch.cuda.is_available())",
        ],
        cwd=str(REPO_ROOT),
        check=True,
    )
else:
    print(
        "跳过 PyTorch 安装。将 INSTALL_PYTORCH 改为 True 可安装 "
        "PyTorch 2.8.0 + CUDA 12.8。"
    )


## 6. 下一步

- **浏览器访问：** 点击上面的链接，或手动打开 `http://127.0.0.1:8765`。
- **导入数据：** 进入“数据集”页，从 AWMC-MaiArchive 在线检索，或上传包含音频和
  `maidata` 的 zip。也可以运行上面的演示数据单元格。
- **推理：** “推理”页需要一个 P3 checkpoint；仓库不附带已训练权重。
- **训练：** 先运行上一节的 PyTorch 安装单元格（PyTorch 2.8.0 + CUDA 12.8），
  它同时会安装 `requirements-train.txt`。训练仍应在 GPU 服务器上进行，
  完整流程见 `docs/TRAINING.md`。

- **停止服务：** 把上一节的 `STOP_WEBUI` 改为 `True` 后运行，或直接结束
  `out/webui-notebook.pid` 中记录的进程。

- **修改监听地址：** 在“启动 Web UI”的 helper 单元格中修改 `BIND_HOST` /
  `ACCESS_HOST`，然后重新运行 helper 和启动单元格。服务没有登录鉴权，使用
  `0.0.0.0` 前请确认网络环境可信。
